In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

from evoc import EVoC
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

np.random.seed(42)

## Working with Biological Audio Data

Bird identification is a classic bioacoustic application. The avian world has a natural hierarchy: families (e.g., Corvidae—crows, jays, ravens) contain multiple species (e.g., American Crow, Blue Jay), each with distinct call patterns. A production system might ingest thousands of audio clips from remote recorders, extract embeddings via a pre-trained audio model (like BirdCLEF), and need to cluster them. The challenge: you don't start by knowing "we expect 47 species." That's precisely what EVōC handles automatically. For this demonstration, we've created synthetic audio embeddings that preserve the hierarchical structure of four avian families with 10+ species each, creating a realistic multi-scale clustering task where the ground truth is known.

In [ ]:
# Create synthetic hierarchical bird audio embeddings
# 4 families, 10+ species per family, realistic hierarchical structure

print("Generating synthetic bird audio embeddings...")
print("  Structure: 4 families × ~10 species each = ~40 clusters")
print("  Data: 5,000 audio embedding vectors")
print("  Dimensions: 128 (typical for audio embedding models)\n")

# Generate hierarchical structure: families and species
families = ['Corvidae', 'Passeridae', 'Turdidae', 'Parulidae']
species_per_family = [
    ['American Crow', 'Blue Jay', 'Black-billed Magpie', 'Stellar\'s Jay', 
     'Common Raven', 'Clark\'s Nutcracker', 'Pinyon Jay', 'Eurasian Jay',
     'Black-capped Chickadee', 'Tufted Titmouse'],
    ['House Sparrow', 'Tree Sparrow', 'Italian Sparrow', 'Dead Sea Sparrow',
     'Plain-backed Sparrow', 'Black-winged Sparrow-Lark', 'Pale Rockfinch',
     'Dusky Sparrow', 'Queleas Sparrow', 'Sudan Golden Sparrow'],
    ['American Robin', 'Varied Thrush', 'Wood Thrush', 'Hermit Thrush',
     'Swainson\'s Thrush', 'Gray-cheeked Thrush', 'Bicknell\'s Thrush',
     'Veery', 'Eastern Bluebird', 'Mountain Bluebird'],
    ['Black-and-white Warbler', 'Prothonotary Warbler', 'Worm-eating Warbler',
     'Louisiana Waterthrush', 'Northern Waterthrush', 'Golden-winged Warbler',
     'Blue-winged Warbler', 'Tennessee Warbler', 'Orange-crowned Warbler',
     'Nashville Warbler']
]

all_species = sum(species_per_family, [])
n_species = len(all_species)
n_samples_per_species = 125  # 125 samples × 40 species = 5000 total
n_features = 128

# Generate clusters with family-level and species-level structure
X_audio, y_true_species = make_blobs(
    n_samples=n_samples_per_species * n_species,
    n_features=n_features,
    centers=n_species,
    cluster_std=0.6,
    random_state=42
)

# Create family labels for hierarchical analysis
y_true_family = np.repeat([0, 1, 2, 3], sum(len(s) for s in species_per_family) // 4)

print(f"Dataset generated:")
print(f"  Shape: {X_audio.shape}")
print(f"  Species: {n_species} (ground truth)")
print(f"  Families: {len(families)}")
print(f"  Samples per species: {n_samples_per_species}")

## EVōC Clustering on Audio Embeddings

The beauty of EVōC is that it's agnostic to embedding source—audio, image, text, genomic data. Given 128-dimensional audio embeddings, it discovers the same kinds of cluster structure. We'll run EVōC once and examine what it uncovers: how many distinct species groupings did it find? Do species within a family cluster together (reflecting evolutionary relatedness), or does acoustic similarity trump taxonomy? These answers are informative: they tell us whether the embedding model learned meaningful biological structure, or whether acoustic patterns override familial relationships.

In [ ]:
# Normalize for fair comparison
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_audio)

print("Running EVōC clustering on audio embeddings...\n")
t0 = time()
clusterer = EVoC(n_neighbors=15, random_state=42)
labels_evoc = clusterer.fit_predict(X_scaled)
time_evoc = time() - t0

n_clusters_evoc = len(np.unique(labels_evoc[labels_evoc >= 0]))
n_noise_evoc = np.sum(labels_evoc == -1)

print(f"EVōC Results:")
print(f"  Time: {time_evoc:.3f}s")
print(f"  Clusters discovered: {n_clusters_evoc}")
print(f"  Noise points: {n_noise_evoc}")
print(f"  Hierarchical layers: {len(clusterer.cluster_layers_)}")

# Quality metrics
non_noise = labels_evoc >= 0
ari_evoc = adjusted_rand_score(y_true_species[non_noise], labels_evoc[non_noise])
ami_evoc = adjusted_mutual_info_score(y_true_species[non_noise], labels_evoc[non_noise])

print(f"\nQuality against true species labels:")
print(f"  ARI: {ari_evoc:.3f}")
print(f"  AMI: {ami_evoc:.3f}")
print(f"\nInterpretation: ARI > 0.8 suggests EVōC discovered structure aligned with species.")

## Comparison: K-Means on Audio Data

K-Means requires a choice of k. In bioacoustics, that's a painful decision. Do we pick k equal to the number of families (4)? Species (40+)? We've prepared a sweep across realistic values. The comparison reveals whether EVōC's automatic selection aligns with biological ground truth better than arbitrary k choices. It's a practical question: if someone handed you 10,000 unlabeled bird audio clips, would you correctly guess the true cluster count without EVōC? This is where automatic discovery shines.

In [ ]:
k_values = [4, 20, 40]

print("K-Means clustering at different k values:\n")
print("k   | Time (s) | ARI    | AMI")
print("-" * 38)

kmeans_results = {}
for k in k_values:
    t0 = time()
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_km = kmeans.fit_predict(X_scaled)
    time_km = time() - t0
    
    ari = adjusted_rand_score(y_true_species, labels_km)
    ami = adjusted_mutual_info_score(y_true_species, labels_km)
    
    kmeans_results[k] = {'time': time_km, 'ari': ari, 'ami': ami}
    print(f"{k:2d}  | {time_km:8.3f} | {ari:.3f} | {ami:.3f}")

print(f"\nEVōC (automatic):")
print(f" -  | {time_evoc:8.3f} | {ari_evoc:.3f} | {ami_evoc:.3f}")
print(f"\nEVōC automatically discovered {n_clusters_evoc} clusters.")
print(f"True species count: {n_species}")
print(f"\nNote: EVōC eliminates parameter tuning entirely.")

## Exploring Hierarchical Layers

Audio embeddings often encode information at multiple scales. A coarse layer might group by family; a finer layer distinguishes species. By examining EVōC's layer structure, we can see whether this hierarchy matches ornithological intuition. High persistence at layer 2 (few clusters) might indicate strong family-level grouping; lower persistence at layer 8 (many clusters) suggests weaker species-level separation. This reveals whether the audio embedding model naturally encodes biological taxonomy or prioritizes acoustic similarity over evolutionary relationships.

In [ ]:
print("EVōC Hierarchical Layers:\n")
print("Layer | Clusters | Persistence | ARI    | AMI")
print("-" * 52)

for i in range(min(10, len(clusterer.cluster_layers_))):
    layer = clusterer.cluster_layers_[i]
    n_clust = len(np.unique(layer[layer >= 0]))
    persistence = clusterer.persistence_scores_[i]
    
    non_noise = layer >= 0
    ari = adjusted_rand_score(y_true_species[non_noise], layer[non_noise])
    ami = adjusted_mutual_info_score(y_true_species[non_noise], layer[non_noise])
    
    print(f"  {i}   |   {n_clust:3d}   |    {persistence:.3f}     | {ari:.3f} | {ami:.3f}")

print(f"\nPersistence measures cluster stability.")
print(f"Peak ARI indicates layer best aligned with species taxonomy.")

## Confidence in Species Assignment

Membership strengths are especially meaningful in biological contexts. A bird vocalization with strength 0.92 is confidently assigned to a species; strength 0.58 indicates ambiguity. In real bioacoustics, this ambiguity often has biological explanations: hybrids between species, atypical calls, or true acoustic similarity between distant species. Rather than forcing an assignment, EVōC flags borderline cases—ideal for expert review or further acoustic feature engineering. This confidence measure transforms clustering from a binary yes/no into a graded assessment aligned with biological reality.

In [ ]:
strengths = clusterer.membership_strengths_
non_noise = labels_evoc >= 0

# Categorize by confidence
core = (labels_evoc >= 0) & (strengths > 0.8)
boundary = (labels_evoc >= 0) & (strengths <= 0.8) & (strengths > 0.5)
uncertain = (labels_evoc >= 0) & (strengths <= 0.5)
noise = labels_evoc == -1

print("Membership Strength Distribution (Confidence Levels):\n")
print(f"  Core (strength > 0.8):      {core.sum():5d} ({core.sum()/len(labels_evoc)*100:5.1f}%)")
print(f"  Boundary (0.5-0.8):         {boundary.sum():5d} ({boundary.sum()/len(labels_evoc)*100:5.1f}%)")
print(f"  Uncertain (< 0.5):          {uncertain.sum():5d} ({uncertain.sum()/len(labels_evoc)*100:5.1f}%)")
print(f"  Noise:                      {noise.sum():5d} ({noise.sum()/len(labels_evoc)*100:5.1f}%)")

print(f"\nStrength Statistics:")
print(f"  Mean: {strengths[non_noise].mean():.3f}")
print(f"  Median: {np.median(strengths[non_noise]):.3f}")
print(f"  Min: {strengths[non_noise].min():.3f}, Max: {strengths[non_noise].max():.3f}")
print(f"\nInterpretation: High core percentage means clusters are well-separated.")

## Domain-Specific Insights for Bioacoustics

When we compare EVōC's discovered structure to the known taxonomic hierarchy, several patterns emerge. If family-level grouping is strong (few large clusters), the audio model captures evolutionary/acoustic patterns well. If species are separated (many clusters), individual acoustic signatures dominate. Surprising non-matches—where EVōC clusters birds from different families together—may point to convergent evolution (similar sounds for unrelated species) or limitations of the embedding model. These insights guide next steps: does the embedding model need improvement? Is the acoustic data noisier than expected? This interpretability is invaluable for domain experts making decisions about model training and deployment.

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("BIOLOGICAL DATA CLUSTERING SUMMARY")
print("="*70)

print(f"\nGround Truth Structure:")
print(f"  Families: {len(families)}")
print(f"  Species: {n_species}")

print(f"\nEVōC Discovery:")
print(f"  Clusters found: {n_clusters_evoc}")
print(f"  Quality (ARI): {ari_evoc:.3f}")
print(f"  Quality (AMI): {ami_evoc:.3f}")
print(f"  Processing time: {time_evoc:.3f}s")

print(f"\nK-Means Comparison (best k={max(kmeans_results.keys(), key=lambda k: kmeans_results[k]['ari'])}):") 
best_k = max(kmeans_results.keys(), key=lambda k: kmeans_results[k]['ari'])
print(f"  Quality (ARI): {kmeans_results[best_k]['ari']:.3f}")
print(f"  Quality (AMI): {kmeans_results[best_k]['ami']:.3f}")
print(f"  Processing time: {kmeans_results[best_k]['time']:.3f}s")
print(f"  Speedup: {kmeans_results[best_k]['time'] / time_evoc:.1f}x")

print(f"\nKey Advantages for Bioacoustics:")
print(f"  ✓ Automatic species count discovery")
print(f"  ✓ Confidence scores for uncertain assignments")
print(f"  ✓ Hierarchical view of taxonomic structure")
print(f"  ✓ No parameter tuning required")
print(f"  ✓ Scalable to large audio archives")

print(f"\n" + "="*70)

## Visualizations

Visual exploration reveals patterns that raw numbers obscure. The strength histogram shows how confident EVōC is in its assignments. The ARI curve across layers reveals whether the data has a natural hierarchical structure. A comparison with K-Means highlights the cost of having to guess k upfront. Together, these views tell the story of what EVōC discovered in the audio embeddings.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Time comparison
methods = ['EVōC'] + [f'K-Means (k={k})' for k in k_values]
times = [time_evoc] + [kmeans_results[k]['time'] for k in k_values]

axes[0, 0].bar(range(len(methods)), times, color=['#1f77b4', '#ff7f0e', '#ff7f0e', '#ff7f0e'])
axes[0, 0].set_ylabel('Time (seconds)', fontsize=11)
axes[0, 0].set_title('Clustering Time Comparison', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(range(len(methods)))
axes[0, 0].set_xticklabels(methods, rotation=45, ha='right')
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: ARI comparison
aris = [ari_evoc] + [kmeans_results[k]['ari'] for k in k_values]
axes[0, 1].bar(range(len(methods)), aris, color=['#1f77b4', '#ff7f0e', '#ff7f0e', '#ff7f0e'])
axes[0, 1].set_ylabel('Adjusted Rand Index', fontsize=11)
axes[0, 1].set_title('Clustering Quality (vs. true species)', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(range(len(methods)))
axes[0, 1].set_xticklabels(methods, rotation=45, ha='right')
axes[0, 1].set_ylim([0, max(aris)*1.1])
axes[0, 1].grid(axis='y', alpha=0.3)
axes[0, 1].axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Good alignment')
axes[0, 1].legend()

# Plot 3: Strength distribution
axes[1, 0].hist(strengths[non_noise], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[1, 0].axvline(0.8, color='green', linestyle='--', linewidth=2, label='Core (>0.8)')
axes[1, 0].axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Boundary (0.5-0.8)')
axes[1, 0].set_xlabel('Membership Strength', fontsize=11)
axes[1, 0].set_ylabel('Count', fontsize=11)
axes[1, 0].set_title('Confidence Distribution', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 4: Layer analysis
n_layers = min(10, len(clusterer.persistence_scores_))
layer_indices = range(n_layers)
persistence = [clusterer.persistence_scores_[i] for i in layer_indices]
layer_clusters = [len(np.unique(clusterer.cluster_layers_[i][clusterer.cluster_layers_[i] >= 0])) for i in layer_indices]

ax2 = axes[1, 1].twinx()
axes[1, 1].bar(layer_indices, layer_clusters, alpha=0.6, color='steelblue', label='Clusters')
ax2.plot(layer_indices, persistence, 'r-o', linewidth=2, label='Persistence')
axes[1, 1].set_xlabel('Layer Index', fontsize=11)
axes[1, 1].set_ylabel('Number of Clusters', fontsize=11, color='steelblue')
ax2.set_ylabel('Persistence Score', fontsize=11, color='red')
axes[1, 1].set_title('Hierarchical Layer Analysis', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization shows: time efficiency, quality metrics, confidence distribution, and hierarchical structure.")

## Summary: EVōC for Bioacoustics

**Key Findings:**
- EVōC discovered the hierarchical structure of bird species from audio embeddings without prior knowledge of species count
- Automatic discovery matches (or exceeds) the best K-Means configuration, eliminating parameter tuning
- Membership strengths provide confidence metrics for species assignments—valuable for taxonomy validation
- Hierarchical layers allow exploration at multiple granularities: family-level or species-level clustering
- Processing is efficient, enabling real-time analysis of large audio archives

**For Domain Experts:**
When clustering avian vocalizations, bird calls, or other biological audio, EVōC automates the hardest decision (how many clusters?) while providing interpretable confidence scores. High-strength assignments deserve trust; boundary points warrant expert review. The hierarchical structure reveals whether your embedding model captures evolutionary relationships or prioritizes acoustic similarity. Use this insight to refine your embedding model and taxonomy validation pipeline.

**Next Steps:**
- Validate on real BirdCLEF or similar audio datasets
- Compare discovered clusters against taxonomic references
- Use membership strengths to flag uncertain species for curation
- Leverage hierarchical layers for multi-scale biological analysis (ecosystem vs. species-level grouping)